# Handwritten Text Recognition

David Hayriyan · PyTorch · EMNIST Balanced / MNIST

This notebook runs the refactored modules from the repository. No training outputs or pretrained model are supplied. Historical results are documented separately in the README.

The setup below installs a CPU build for portability. For GPU training, install the matching official CUDA build instead before running the remaining cells. The notebook has not been executed in Colab during project preparation.

In [ ]:
from pathlib import Path
import subprocess
import sys

repo = Path('/content/handwritten-text-recognition')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/David-Hayriyan/handwritten-text-recognition..git', str(repo)], check=True)
%cd /content/handwritten-text-recognition
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.14.1', 'torchvision==0.29.1', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.txt'], check=True)

## Check the environment

These checks use synthetic data. They do not measure recognition quality.

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)

## Train

The official training split is partitioned into train and validation. The official test split is reserved. Change DATASET to mnist for digits only. A fresh run directory is generated to preserve earlier runs. Dataset download and training can take time.

In [ ]:
from datetime import datetime, timezone
DATASET = 'emnist-balanced'
RUN = 'runs/' + DATASET + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
command = [sys.executable, '-m', 'htr.train', '--dataset', DATASET, '--download', '--epochs', '30', '--patience', '5', '--output', RUN]
subprocess.run(command, check=True)

## Learning curves

In [ ]:
subprocess.run([sys.executable, '-m', 'htr.plot_history', '--run', RUN], check=True)
from IPython.display import display, Image
display(Image(filename=RUN + '/learning_curves.png'))

## Final test evaluation

Run only after freezing the configuration and selecting a checkpoint using validation. Do not use these results to choose hyperparameters.

In [ ]:
EVAL = RUN + '-test'
subprocess.run([sys.executable, '-m', 'htr.evaluate', '--checkpoint', RUN + '/best.pt', '--download', '--output', EVAL], check=True)
display(Image(filename=EVAL + '/confusion_matrix.png'))

## Predict your own image

Upload one upright, isolated character on a plain background. The model reports softmax scores, which are not calibrated confidence estimates.

In [ ]:
from google.colab import files
from htr.predict import predict
import json

uploaded = files.upload()
if uploaded:
    filename = next(iter(uploaded))
    result = predict(RUN + '/best.pt', filename, top_k=5, save_preprocessed=RUN + '/user-input.png')
    print(json.dumps(result, indent=2))
    display(Image(filename=RUN + '/user-input.png'))

## Save your work

Colab storage is temporary. Download your run before the session ends. The archive may contain your private input image; review it before sharing. No run data is automatically uploaded to GitHub.

In [ ]:
import shutil
archive = shutil.make_archive(RUN, 'zip', root_dir=RUN)
files.download(archive)

## Optional hyperparameter study

The following eight-trial grid uses only training and validation data. It is commented out because it requires multiple full training runs. Choose a new output folder if it already exists.

In [ ]:
# subprocess.run([sys.executable, '-m', 'htr.experiments', '--dataset', DATASET, '--download', '--epochs', '30', '--output', 'runs/grid'], check=True)